# Session 9: Building Agents with the OpenAI Agents SDK

This notebook builds progressively more capable agents using the OpenAI Agents SDK:

1. **Hello Agent** — the simplest agent: no tools, just instructions
2. **Tool-using agent** — add a calculator, date tool, and FAISS search
3. **Inspect the trace** — see exactly what the agent did step by step
4. **Orchestrator-workers** — one agent delegates to specialists
5. **Handoffs** — transfer control permanently to another agent
6. **Reflection exercise** — tool descriptions matter more than you think

### Core SDK primitives

| Primitive | Description |
|---|---|
| `Agent` | An LLM with instructions, tools, and optional handoffs |
| `Runner` | Executes an agent and manages the agentic loop |
| `function_tool` | Decorator that exposes a Python function as an agent tool |
| `handoff` | One-way transfer of control to another agent |

In [ ]:
import os
import math
import numpy as np
import faiss
from datetime import date
from openai import OpenAI
from agents import Agent, Runner, function_tool, handoff
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
print("Setup complete — OpenAI Agents SDK ready")

## 1. Hello Agent — The Simplest Case

An `Agent` is an LLM with instructions. `Runner.run_sync()` runs it and
manages the loop until the agent produces a final output.

In [ ]:
# The simplest possible agent: no tools, just instructions + model
hello_agent = Agent(
    name="Course Assistant",
    instructions=(
        "You are a helpful assistant for an applied NLP course. "
        "Be concise — answer in 2–3 sentences."
    ),
    model="gpt-4o-mini",
)

result = Runner.run_sync(hello_agent, "What is the difference between a workflow and an agent?")
print(result.final_output)

### What does `result` contain?

In [ ]:
print(f"Type:          {type(result).__name__}")
print(f"Last agent:    {result.last_agent.name}")
print(f"Steps taken:   {len(result.new_items)}")
print()
for item in result.new_items:
    print(f"  {item.type}")

## 2. Adding Tools

Tools are Python functions decorated with `@function_tool`.
The docstring becomes the tool description that the agent reads to decide
when and how to use the tool.

### Structured output in the Agents SDK — two places it appears

The `BaseModel` pattern from Session 8 appears in two distinct places in the SDK:

**1. Tool return types** — a tool function returns a Pydantic model instead of a string.
The agent receives a typed, structured result it can rely on, rather than text it has to interpret.

**2. `output_type` on `Agent`** — constrains the agent's *final answer* to a Pydantic schema.
```python
agent = Agent(output_type=MyModel, ...)
result = Runner.run_sync(agent, "question")
result.final_output   # MyModel instance — not a string
```
The SDK passes the schema to the LLM and validates the response.
`result.final_output` is then a typed Python object — fields are directly accessible,
no parsing required. The agent can still call tools freely; only the final answer must match the schema.

We use both in this notebook:
- `calculate` returns a `CalcResult` model (tool return type)
- `researcher` agent uses `output_type=ResearchSummary` (agent output type — shown later)

We define three tools:
- `calculate` — returns a `CalcResult` Pydantic model (structured output)
- `get_todays_date` — returns a plain string (simple enough to not need a model)
- `search_course_materials` — returns formatted string passages

In [ ]:
from pydantic import BaseModel

class CalcResult(BaseModel):
    expression: str
    result: float
    formatted: str  # human-readable: "0.85 ** 10 = 0.1969"

@function_tool
def calculate(expression: str) -> CalcResult:
    """
    Evaluate a mathematical expression safely.
    Supports: +, -, *, /, **, sqrt, log, sin, cos, pi, e.
    Examples: '0.85 ** 10', 'math.sqrt(144)', '(3 + 4) * 2'
    Returns a structured result with the expression, numeric value, and formatted string.
    """
    safe_env = {k: getattr(math, k) for k in dir(math) if not k.startswith("_")}
    safe_env["__builtins__"] = {}
    try:
        value = float(eval(expression, safe_env))
        return CalcResult(
            expression=expression,
            result=value,
            formatted=f"{expression} = {value}",
        )
    except Exception as e:
        return CalcResult(expression=expression, result=0.0, formatted=f"Error: {e}")

@function_tool
def get_todays_date() -> str:
    """Return today's date in YYYY-MM-DD format."""
    return str(date.today())

# Test tools directly before giving them to an agent
result_calc = calculate("0.85 ** 10")
print(type(result_calc))          # CalcResult — a typed object, not a string
print(result_calc.result)         # 0.1968...  — directly accessible field
print(result_calc.formatted)      # "0.85 ** 10 = 0.1968..."
print(get_todays_date())

In [ ]:
# Build a small in-memory FAISS index over course content
# (Same texts as notebook 01 — in a real app you'd load this from disk)
COURSE_TEXTS = [
    "Retrieval-Augmented Generation (RAG) combines a retrieval system with an LLM to ground answers in documents.",
    "FAISS is a library for efficient similarity search over dense vectors, built by Meta.",
    "Cosine similarity measures the angle between two vectors, returning a value between -1 and 1.",
    "An embedding is a dense numerical representation of text that captures semantic meaning.",
    "BM25 is a lexical search algorithm scoring by term frequency and inverse document frequency.",
    "The OpenAI Agents SDK uses Agent, Runner, function_tool, and handoffs as its core primitives.",
    "Agentic RAG lets an LLM iteratively retrieve and refine its answer rather than retrieving once.",
    "MCP (Model Context Protocol) is an open standard for connecting AI models to tools and data sources.",
    "Prompt chaining passes the output of one LLM call as the input to the next.",
    "The compound accuracy problem: a 10-step workflow at 85% accuracy per step succeeds only 20% of the time.",
]

def _embed(texts: list[str]) -> np.ndarray:
    response = client.embeddings.create(model="text-embedding-3-small", input=texts)
    return np.array([item.embedding for item in response.data], dtype=np.float32)

_course_vecs = _embed(COURSE_TEXTS)
_course_index = faiss.IndexFlatL2(_course_vecs.shape[1])
_course_index.add(_course_vecs)
print(f"Course index ready: {_course_index.ntotal} vectors")

@function_tool
def search_course_materials(query: str) -> str:
    """
    Search ANLP course materials for information relevant to the query.
    Returns the 2 most relevant passages.
    Use this for questions about course concepts, terminology, and techniques.
    """
    q_vec = _embed([query]).reshape(1, -1)
    _, indices = _course_index.search(q_vec, 2)
    passages = [COURSE_TEXTS[i] for i in indices[0]]
    return "\n\n".join(f"[{i+1}] {p}" for i, p in enumerate(passages))

# Test the search tool
print(search_course_materials("how does semantic search work?"))

### Agent with all three tools

The agent now decides which tools to call — and in what order — based on the question.

In [ ]:
tool_agent = Agent(
    name="Course Assistant",
    instructions="""You are a helpful assistant for an applied NLP course.
    - Use search_course_materials for questions about course content and concepts
    - Use calculate for any mathematical computation
    - Use get_todays_date when the current date is relevant
    Always explain your reasoning and which tools you used.""",
    tools=[search_course_materials, calculate, get_todays_date],
    model="gpt-4o-mini",
)

result = Runner.run_sync(
    tool_agent,
    "What is cosine similarity, and if each step in a 10-step pipeline is 85% accurate, "
    "what is the overall success rate?",
)
print(result.final_output)

## 3. True Agentic RAG — The LLM Controls the Loop

In notebook 01 we built a manual RAG loop: the programmer decided when to retrieve,
when to check completeness, when to rewrite the query.

Now we give the LLM a `retrieve` tool and let `Runner` manage the loop.
The LLM decides:
- **if** it needs to retrieve
- **what** query to use
- **how many times** to retrieve
- **when** it has enough context to answer

Same outcome. Different control. This is the difference between a workflow and an agent.

In [ ]:
@function_tool
def retrieve(query: str) -> str:
    """
    Search ANLP course materials for information relevant to the query.
    Returns the 3 most relevant passages.
    Call this whenever you need factual grounding before answering.
    You decide when to call it, what to search for, and whether to call it again.
    """
    q_vec = _embed([query]).reshape(1, -1)
    _, indices = _course_index.search(q_vec, 3)
    passages = [COURSE_TEXTS[i] for i in indices[0]]
    return "\n\n".join(f"[{i+1}] {p}" for i, p in enumerate(passages))

rag_agent = Agent(
    name="RAG Agent",
    instructions=(
        "Answer questions by searching course materials first. "
        "Search as many times as needed with different queries until you have enough information. "
        "Only produce a final answer when you are confident it is complete and accurate."
    ),
    tools=[retrieve],
    model="gpt-4o-mini",
)

rag_result = Runner.run_sync(
    rag_agent,
    "How does FAISS support semantic search and what similarity metric does it use?",
)
print(rag_result.final_output)

### What did the agent decide?

Inspect the trace to see the LLM's decisions — queries it chose, number of retrievals,
when it decided it had enough context.

In [ ]:
print(f"Steps the agent took: {len(rag_result.new_items)}\n")

for i, item in enumerate(rag_result.new_items, 1):
    if item.type == "tool_call_item":
        import json
        args = json.loads(item.raw_item.arguments)
        print(f"Step {i} — RETRIEVE")
        print(f"  Query the LLM chose: {args.get('query', '')}")
    elif item.type == "tool_call_output_item":
        output = str(item.raw_item.get("output", ""))
        print(f"Step {i} — Retrieved {len(output)} chars of context")
    elif item.type == "message_output_item":
        print(f"Step {i} — LLM decided it had enough → produced final answer")
    print()

### Workflow vs. Agent — Side by Side

| | Manual `agentic_rag` (notebook 01) | `rag_agent` (this notebook) |
|---|---|---|
| Who decides to retrieve? | Programmer's `for` loop | LLM, via tool call |
| Who decides when to stop? | Programmer's `is_complete()` | LLM producing final output |
| Who decides what to search? | Programmer's `rewrite_query()` | LLM choosing its own query |
| Max iterations enforced by | `max_iterations` parameter | `Runner` (default: 10 turns) |
| Control flow lives in | Your Python code | The model's reasoning |

The agent's queries are different from ours. Its number of retrievals varies by question.
It stops when *it* is satisfied — not when our code tells it to stop.

This is what "model-directed control flow" means in practice.